# 07 — SVM Classification (Demand Level t+1)

**Member 4 — Infrastruktur, SVM & Optimasi**

Tujuan: membandingkan Linear SVM dan RBF SVM untuk memprediksi `demand_level` pada periode berikutnya.

> Dataset belum tersedia di repository saat notebook ini dibuat. Notebook ini adalah template pipeline; training baru dapat dijalankan setelah file hasil feature engineering tersedia.

**Aturan eksperimen**
- Split harus kronologis: Train (Jan–Apr), Validation (Mei), Test (Juni).
- Scaling (`StandardScaler`) di-fit hanya pada Train.
- Test set tidak digunakan untuk tuning.
- Untuk mengatasi ukuran data besar, SVM dapat dilatih menggunakan sampel dari Train saja. Validation tetap tidak disampling untuk evaluasi.
- Catat training time dan prediction time.


In [ ]:
# 1. Imports & reproducibility
import os, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

RANDOM_STATE = 42
OUT_DIR = Path("../out")  # adjust if notebook location differs
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Imports ready; random seed:", RANDOM_STATE)


## 2. Load data

Adjust the path and filenames to match the team's final feature dataset. Prefer separate chronological files for train/validation/test. If there is only one dataset, split it chronologically—not randomly.

Expected target column: `demand_level`.

In [ ]:
# TODO: update these paths/filenames once the feature dataset is shared
TRAIN_PATH = Path("../datasets/processed/train_features.parquet")
VALID_PATH = Path("../datasets/processed/validation_features.parquet")
TEST_PATH  = Path("../datasets/processed/test_features.parquet")

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

if TRAIN_PATH.exists() and VALID_PATH.exists():
    train_df = read_table(TRAIN_PATH)
    val_df = read_table(VALID_PATH)
    test_df = read_table(TEST_PATH) if TEST_PATH.exists() else None
    print("Train:", train_df.shape, "Validation:", val_df.shape,
          "Test:", None if test_df is None else test_df.shape)
else:
    print("DATA FILES NOT FOUND — update paths after the dataset is shared.")


In [ ]:
# 3. Inspect schema (run after data is loaded)
if "train_df" in globals():
    display(train_df.head())
    print(train_df.dtypes)
    print("Target distribution:")
    display(train_df["demand_level"].value_counts(dropna=False))


## 4. Prepare features

Remove target and any non-feature/time identifier columns. **Review this list manually** to ensure no post-pickup/target leakage fields are included. The timestamp may be retained separately for ordering but should not be passed to SVM as a raw datetime.

In [ ]:
# TODO: explicitly configure columns based on final feature schema
TARGET = "demand_level"
DROP_COLS = [
    TARGET,
    # Add timestamp, zone identifiers, or non-feature columns here as needed.
]

if "train_df" in globals():
    feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
    X_train_full = train_df[feature_cols].copy()
    y_train_full = train_df[TARGET].copy()
    X_val = val_df[feature_cols].copy()
    y_val = val_df[TARGET].copy()

    # Basic guardrails
    assert set(y_train_full.dropna().unique()) <= {"low", "medium", "high"}, \
        "Check demand_level labels; update expected labels if team uses different names."
    print("Feature count:", len(feature_cols))
    print("Features:", feature_cols)


## 5. Sampling strategy for training

SVM can be expensive on millions of rows. Sample **only the training partition** for the first experiment. Validation remains untouched so metrics reflect the full validation set.

For the initial run, use a capped stratified sample to preserve class proportions. If the dataset is already small, use all training rows.

In [ ]:
from sklearn.model_selection import train_test_split

MAX_TRAIN_ROWS = 100_000
MAX_RBF_ROWS = 30_000

if "X_train_full" in globals():
    if len(X_train_full) > MAX_TRAIN_ROWS:
        X_train, _, y_train, _ = train_test_split(
            X_train_full, y_train_full,
            train_size=MAX_TRAIN_ROWS,
            random_state=RANDOM_STATE,
            stratify=y_train_full
        )
    else:
        X_train, y_train = X_train_full, y_train_full

    print("Full train:", len(X_train_full), "| sampled train:", len(X_train),
          "| validation:", len(X_val))
    print("Sample class distribution:")
    display(y_train.value_counts(normalize=True))


## 6. Train & evaluate Linear SVM

`StandardScaler` is inside the Pipeline, so it is fit on the sampled training data only. Evaluate on the complete validation partition.

In [ ]:
def evaluate_model(name, model, Xtr, ytr, Xva, yva):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    train_seconds = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = model.predict(Xva)
    predict_seconds = time.perf_counter() - t1

    metrics = {
        "model": name,
        "accuracy": accuracy_score(yva, pred),
        "precision_macro": precision_score(yva, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(yva, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(yva, pred, average="macro", zero_division=0),
        "train_seconds": train_seconds,
        "predict_seconds": predict_seconds,
        "n_train": len(Xtr),
        "n_validation": len(Xva),
    }
    return metrics, pred

if "X_train" in globals():
    linear_svm = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))
    ])
    linear_metrics, linear_pred = evaluate_model(
        "Linear SVM", linear_svm, X_train, y_train, X_val, y_val
    )
    print(linear_metrics)
    print(classification_report(y_val, linear_pred, zero_division=0))


In [ ]:
# Linear SVM confusion matrix
if "linear_pred" in globals():
    labels = sorted(pd.Series(y_val).dropna().unique())
    cm = confusion_matrix(y_val, linear_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(values_format="d")
    plt.title("Linear SVM — Validation Confusion Matrix")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "svm_linear_confusion_matrix.png", dpi=150)
    plt.show()


## 7. Train & evaluate RBF SVM

RBF SVM has higher computational cost. Limit it to a smaller training-only sample. If runtime is excessive, reduce `MAX_RBF_ROWS` further and document the chosen sample size.

In [ ]:
if "X_train_full" in globals():
    if len(X_train_full) > MAX_RBF_ROWS:
        X_rbf, _, y_rbf, _ = train_test_split(
            X_train_full, y_train_full,
            train_size=MAX_RBF_ROWS,
            random_state=RANDOM_STATE,
            stratify=y_train_full
        )
    else:
        X_rbf, y_rbf = X_train_full, y_train_full

    rbf_svm = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))
    ])

    rbf_metrics, rbf_pred = evaluate_model(
        "RBF SVM", rbf_svm, X_rbf, y_rbf, X_val, y_val
    )
    print(rbf_metrics)
    print(classification_report(y_val, rbf_pred, zero_division=0))


In [ ]:
# RBF confusion matrix
if "rbf_pred" in globals():
    labels = sorted(pd.Series(y_val).dropna().unique())
    cm = confusion_matrix(y_val, rbf_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(values_format="d")
    plt.title("RBF SVM — Validation Confusion Matrix")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "svm_rbf_confusion_matrix.png", dpi=150)
    plt.show()


## 8. Compare models & export metrics

Only export the models that completed successfully. Metrics below are validation metrics; test data must remain untouched until final model selection.

In [ ]:
results = []
if "linear_metrics" in globals():
    results.append(linear_metrics)
if "rbf_metrics" in globals():
    results.append(rbf_metrics)

if results:
    metrics_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
    display(metrics_df)
    metrics_df.to_csv(OUT_DIR / "svm_validation_metrics.csv", index=False)
    print("Saved:", OUT_DIR / "svm_validation_metrics.csv")
else:
    print("No model results yet.")


## 9. Discussion notes (fill after experiments)

- **Sampling:** Explain the training sample cap, seed, and why it was necessary.
- **Scaling:** StandardScaler fit only on training partition because SVM is scale-sensitive.
- **Class imbalance:** `class_weight='balanced'` compensates for unequal class frequencies.
- **Comparison:** Discuss macro-F1, precision/recall, confusion matrix, and training/prediction time.
- **Limitations:** Sampled training may not represent the complete training distribution; RBF training can be computationally expensive.
- **Leakage check:** Confirm no post-trip information, target-derived information, or test data was used during fitting/tuning.
